# Mid-term Project


# Baseline 

Data Manipualation 

In [442]:
import pandas as pd
import os 
import numpy as np

Data Vizualization 

In [443]:
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px

ML & Preprocessing 

In [444]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, accuracy_score
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import xgboost as xgb
from tabulate import tabulate 
from joblib import dump, load
from pathlib import Path


Loaded Dataset

In [445]:
DATA_PATH = Path('WA_Fn-UseC_-Telco-Customer-Churn.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(f"not found {DATA_PATH}")
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Loaded data: {df.shape}")

Loaded data: (7043, 21)


In [406]:
print("===Data Information===")
print(df.info())
print("\n")
print("===First 5 Rows===")
display(df.head())
print('====Dataset Shape===')
print(df.shape)



===Data Information===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7038 non-null   float64
 3   Partner           7043 non-null   str    
 4   Dependents        7042 non-null   str    
 5   tenure            7042 non-null   float64
 6   PhoneService      7042 non-null   str    
 7   MultipleLines     7042 non-null   str    
 8   InternetService   7042 non-null   str    
 9   OnlineSecurity    7042 non-null   str    
 10  OnlineBackup      7042 non-null   str    
 11  DeviceProtection  7042 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-nul

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0.0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0.0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


====Dataset Shape===
(7043, 21)


In [308]:
df.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

# Here is the "Churn" that we're going to take as a target column.

In [309]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       5
Partner             0
Dependents          1
tenure              1
PhoneService        1
MultipleLines       1
InternetService     1
OnlineSecurity      1
OnlineBackup        1
DeviceProtection    1
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

# Features and target 

In [446]:
X = df.drop('Churn', axis=1)
y = df['Churn']

# Split the dataset into X_train and X_test

In [447]:
# train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=9
)


# Pereprocessing 

In [448]:
# Handling Missing Values 
def tozala_train(df):
    fill_values = {}
    for col in df.columns:
        if df[col].dtype == 'str':
            fill_values[col] = df[col].mode()[0]
            df[col] = df[col].fillna(fill_values[col], inplace=True)
        else:
            fill_values[col] = df[col].mean()
            df[col] = df[col].fillna(fill_values[col], inplace=True)
    return df, fill_values

In [449]:
# For the test 
def tozala_test(df, fill_values):
    for col, value in fill_values.items():
        df[col] = df[col].fillna(value)
    return df


In [450]:
# For the implementation
X_train, fill_values = tozala_train(X_train)
X_test = tozala_test(X_test, fill_values)

/var/folders/j4/qzhh77r553z1scwnh3mxkx200000gn/T/ipykernel_4346/2904627001.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col] = df[col].fillna(fill_values[col], inplace=True)
/var/folders/j4/qzhh77r553z1scwnh3mxkx200000gn/T/ipykernel_4346/2904627001.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame

In [391]:
X_train.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

# Encoding

In [451]:
def encodla_train(df, one_hot_threshold=5):
    label_encoders = {}
    one_hot_cols = {}

    for col in df.columns:
        if df[col].dtype=='str':
            if df[col].nunique() <= one_hot_threshold:
                one_hot_cols[col] = df[col].unique()
            else:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col])
                label_encoders[col] = le
    # one hot in train 
    df = pd.get_dummies(df, columns=one_hot_cols.keys(), drop_first=True, dtype=int)
    return df, label_encoders, one_hot_cols


In [452]:
# For testing 
def encodla_test(df, label_encoders, one_hot_cols, train_columns):
    for col, le in label_encoders.items():
        df[col] = df[col].apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )
    df = pd.get_dummies(df, columns=one_hot_cols.keys(), drop_first = True, dtype=int)
    # Align columns 
    df = df.reindex(columns=train_columns, fill_value=0)
    return df

In [453]:
# For the implementation
X_train, label_encoders,one_hot_cols = encodla_train(X_train)
X_test = encodla_test(
    X_test, 
    label_encoders,
    one_hot_cols,
    train_columns = X_train.columns
)

In [454]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 6781 to 382
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   customerID                             5634 non-null   int64  
 1   SeniorCitizen                          5634 non-null   float64
 2   tenure                                 5634 non-null   float64
 3   MonthlyCharges                         5634 non-null   float64
 4   TotalCharges                           5634 non-null   int64  
 5   gender_Male                            5634 non-null   int64  
 6   Partner_Yes                            5634 non-null   int64  
 7   Dependents_Yes                         5634 non-null   int64  
 8   PhoneService_Yes                       5634 non-null   int64  
 9   MultipleLines_No phone service         5634 non-null   int64  
 10  MultipleLines_Yes                      5634 non-null   int64  
 11  InternetService_Fi

In [345]:
X_test.info()

<class 'pandas.DataFrame'>
Index: 1409 entries, 1396 to 1952
Data columns (total 31 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   customerID                             1409 non-null   int64  
 1   SeniorCitizen                          1409 non-null   float64
 2   tenure                                 1409 non-null   float64
 3   MonthlyCharges                         1409 non-null   float64
 4   TotalCharges                           1409 non-null   int64  
 5   gender_Male                            1409 non-null   int64  
 6   Partner_Yes                            1409 non-null   int64  
 7   Dependents_Yes                         1409 non-null   int64  
 8   PhoneService_Yes                       1409 non-null   int64  
 9   MultipleLines_No phone service         1409 non-null   int64  
 10  MultipleLines_Yes                      1409 non-null   int64  
 11  InternetService_F

In [455]:
# On below, we can see the error which is that y_train and y_test should be encoded from ['No', 'Yes'] to [0,1]
y_train = y_train.map({'No': 0, 'Yes': 1})
y_test = y_test.map({'No': 0, 'Yes': 1})

# Scaling 

In [416]:
def scaleqil_train(df):
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns

    # Initialize scaler and fit on train 
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    return df, scaler, num_cols


In [417]:
# For Test Scaling
def scaleqil_test(df, scaler, num_cols):
    df[num_cols] = scaler.transform(df[num_cols])
    return df

In [418]:
# For the implementation 
X_train, scaler, num_cols = scaleqil_train(X_train)
X_test = scaleqil_test(
    X_test, 
    scaler,
    num_cols
)

In [280]:
X_train.head()

,customerID,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
6781,-0.016909,-0.440681,-1.200066,0.156693,-0.754870,0.979966,-0.965796,1.527654,0.325587,-0.325587,...,-0.523530,-0.789703,-0.523530,-0.795343,-0.517747,-0.559447,-1.203537,-0.526281,1.413649,-0.546859
2777,-0.037814,-0.440681,-0.914494,-1.493826,-0.982050,0.979966,-0.965796,-0.654598,0.325587,-0.325587,...,1.910109,-0.789703,1.910109,-0.795343,-0.517747,-0.559447,-1.203537,-0.526281,1.413649,-0.546859
4857,0.482355,-0.440681,-0.506535,-1.330935,0.547977,-1.020443,-0.965796,-0.654598,-3.071373,3.071373,...,-0.523530,-0.789703,-0.523530,-0.795343,-0.517747,-0.559447,-1.203537,1.900125,-0.707389,-0.546859
3833,1.453828,-0.440681,-0.832902,0.326233,1.521512,0.979966,-0.965796,-0.654598,0.325587,-0.325587,...,-0.523530,1.266299,-0.523530,1.257319,-0.517747,-0.559447,-1.203537,-0.526281,-0.707389,-0.546859
4170,-1.436612,-0.440681,1.614855,1.669253,1.541808,0.979966,1.035416,-0.654598,0.325587,-0.325587,...,-0.523530,1.266299,-0.523530,1.257319,-0.517747,1.787480,-1.203537,-0.526281,-0.707389,-0.546859


In [281]:
X_test.head()

,customerID,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
1396,-1.732358,-0.440681,1.288487,1.167283,-1.723822,-1.020443,-0.965796,-0.654598,0.325587,-0.325587,...,-0.523530,1.266299,-0.523530,1.257319,1.931445,-0.559447,0.830884,1.900125,-0.707389,-0.546859
4589,-1.732358,-0.440681,-0.914494,-1.488839,-1.723822,-1.020443,1.035416,1.527654,0.325587,-0.325587,...,1.910109,-0.789703,1.910109,-0.795343,-0.517747,1.787480,-1.203537,-0.526281,-0.707389,1.828625
6846,-1.732358,-0.440681,-1.281658,0.485799,1.446222,-1.020443,-0.965796,-0.654598,0.325587,-0.325587,...,-0.523530,-0.789703,-0.523530,-0.795343,-0.517747,-0.559447,0.830884,-0.526281,-0.707389,1.828625
4045,-1.732358,-0.440681,-0.751310,0.201571,-1.723822,-1.020443,-0.965796,-0.654598,0.325587,-0.325587,...,-0.523530,-0.789703,-0.523530,-0.795343,-0.517747,-0.559447,0.830884,1.900125,-0.707389,-0.546859
1885,-1.732358,-0.440681,-0.343351,0.178301,-1.723822,0.979966,1.035416,1.527654,0.325587,-0.325587,...,-0.523530,-0.789703,-0.523530,1.257319,-0.517747,-0.559447,0.830884,1.900125,-0.707389,-0.546859


# Algorithms need to be scaled

In [420]:
# Using Pipelines and joblib to increase performance
from joblib import Parallel, delayed
pipelines = {
    "LR": LogisticRegression(random_state=9),
    "SVM": SVC(kernel='linear', C=1.0),
    "KNN": KNeighborsClassifier()
}

def train_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return name, model, y_pred

results_scaled = Parallel(n_jobs=-1)(
    delayed(train_model)(name, model)
    for name, model in pipelines.items()
)
# Results 
for name, model, y_pred in results_scaled:
    print(f"\n==={name}===")
    print(f"Classification Report: \n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred)}")


===LR===
Classification Report: 
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1044
           1       0.65      0.53      0.58       365

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.80      1409

Confusion Matrix: 
[[938 106]
 [171 194]]

===SVM===
Classification Report: 
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1044
           1       0.65      0.53      0.59       365

    accuracy                           0.81      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix: 
[[941 103]
 [171 194]]

===KNN===
Classification Report: 
              precision    recall  f1-score   support

           0       0.83      0.85      0.84      1044
           1       0.53      0.48      0.51      

# Comparison model results 

In [425]:
comparison = []
for name, model, y_pred in results_scaled:
    comparison.append({
        "Model" : name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4)
    })
comparison_df = pd.DataFrame(comparison).sort_values(by='Recall', ascending=False)
print(comparison_df.to_string(index=False))

Model  Accuracy  Recall
   LR    0.8034  0.5315
  SVM    0.8055  0.5315
  KNN    0.7559  0.4849


# ALgorithms no need to be scaled

In [378]:
# y_train va y_test ni encode qiling:
y_train = y_train.map({'No': 0, 'Yes': 1})
y_test = y_test.map({'No': 0, 'Yes': 1})

In [456]:
pipelines = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200,
        max_depth = 3,
        learning_rate = 0.01,
        subsample = 1,
        colsample_bytree = 0.8,
        random_state=9
    )
}

def train_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return name, model, y_pred

results_no_scaled = Parallel(n_jobs=-1)(
    delayed(train_model)(name, model)
    for name, model in pipelines.items()
)
# Results 
for name, model, y_pred in results_no_scaled:
    print(f"\n==={name}===")
    print(f"Classification Report: \n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred)}")


===Decision Tree===
Classification Report: 
              precision    recall  f1-score   support

           0       0.78      0.81      0.79      1044
           1       0.38      0.33      0.35       365

    accuracy                           0.69      1409
   macro avg       0.58      0.57      0.57      1409
weighted avg       0.67      0.69      0.68      1409

Confusion Matrix: 
[[849 195]
 [246 119]]

===Random Forest===
Classification Report: 
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1044
           1       0.60      0.47      0.53       365

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.69      1409
weighted avg       0.77      0.78      0.77      1409

Confusion Matrix: 
[[930 114]
 [192 173]]

===XGBoost===
Classification Report: 
              precision    recall  f1-score   support

           0       0.81      0.95      0.87      1044
           1       0.70 

# Comparison results among six algorithms 

In [458]:
comparison_no_scaled = []
for name, model, y_pred in results_no_scaled:
    comparison_no_scaled.append({
        "Model" : name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4)
    })
comparison_df_no_sclaled = pd.DataFrame(comparison_no_scaled).sort_values(by='Recall', ascending=False)
print(comparison_df_no_sclaled.to_string(index=False))

        Model  Accuracy  Recall
Random Forest    0.7828  0.4740
      XGBoost    0.7949  0.3616
Decision Tree    0.6870  0.3260


# Tabulate 

In [459]:
from tabulate import tabulate

baseline = [
    ['Decision Tree Classifier',0.6870, 0.3260 ],
    ['Random Forest Classifier', 0.7828, 0.4740],
    ['XGBClassifier',0.7949, 0.3616],
    ['Logistic Regression',0.8034, 0.5315],
    ['Support Vector Machine',0.8055, 0.5315],
    ['KNN', 0.7559, 0.4849]
    
]
headers = ['Models', 'Accuracy', 'Recall']
table = tabulate(baseline, headers=headers, tablefmt='grid', floatfmt='.4f')
print('\n===============================================================================')
print(table)
print('===============================================================================')


+--------------------------+------------+----------+
| Models                   |   Accuracy |   Recall |
+==========================+============+==========+
| Decision Tree Classifier |     0.6870 |   0.3260 |
+--------------------------+------------+----------+
| Random Forest Classifier |     0.7828 |   0.4740 |
+--------------------------+------------+----------+
| XGBClassifier            |     0.7949 |   0.3616 |
+--------------------------+------------+----------+
| Logistic Regression      |     0.8034 |   0.5315 |
+--------------------------+------------+----------+
| Support Vector Machine   |     0.8055 |   0.5315 |
+--------------------------+------------+----------+
| KNN                      |     0.7559 |   0.4849 |
+--------------------------+------------+----------+


# Best Model with adequate evaluation metrics

In [464]:
best_score = max(model[1] for model in baseline)  # this takes the maximum values
best_models = []  # we will store the best models here

for model in baseline:
    # if model[1] equals the best score and model name matches
    if model[1] == best_score and model[0] in ['Support Vector Machine']:
        # append to best_models list
        best_models.append(model)

for model in best_models:
    print(f'The Best model is {model[0]}, the best recall is {model[1]} which is 80.55%')

The Best model is Support Vector Machine, the best recall is 0.8055 which is 80.55%


# Model Improvement  

In [388]:
# Data Manipualtion
import pandas as pd
import os 
import numpy as np

# Data Vizualization 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px

# ML & Preprocessing 
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, accuracy_score
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import xgboost as xgb
from tabulate import tabulate 
from joblib import dump, load
from pathlib import Path

In [389]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [390]:
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [391]:
df.drop("customerID", axis=1, inplace=True)

In [392]:
df["yearly_charges"] = df["MonthlyCharges"] * 12


In [393]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({
    "No": 0,
    "Yes":1
})

Split into train and test

In [394]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Engineering

In [177]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7038 non-null   float64
 2   Partner           7043 non-null   str    
 3   Dependents        7042 non-null   str    
 4   tenure            7042 non-null   float64
 5   PhoneService      7042 non-null   str    
 6   MultipleLines     7042 non-null   str    
 7   InternetService   7042 non-null   str    
 8   OnlineSecurity    7042 non-null   str    
 9   OnlineBackup      7042 non-null   str    
 10  DeviceProtection  7042 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

Drop unnecessary clumns and create some useful column(s)

# Preprocessing

In [395]:
class Preprocessing:
    def __init__(self, one_hot_threshold=5):
        self.one_hot_threshold = one_hot_threshold
        self.fill_valeus = {}
        self.label_encoders = {}
        self.one_hot_cols = {}
        self.train_columns = None
        self.scaler = StandardScaler()
        self.num_cols = []

    def _encodla_train(self, df):
        for col in df.columns:
            if df[col].dtype == 'str':
                if df[col].nunique() <= self.one_hot_threshold:
                    self.one_hot_cols[col] = df[col].unique()
                else:
                    le = LabelEncoder()
                    df[col] = le.fit_transform(df[col])
                    self.label_encoders[col] = le
        df = pd.get_dummies(df, columns=self.one_hot_cols.keys(), drop_first=True, dtype=int)
        return df
    def _encodla_test(self, df):
        for col, le in self.label_encoders.items():
            df[col] = df[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )
        df = pd.get_dummies(df, columns=self.one_hot_cols.keys(), drop_first=True, dtype=int)
        df = df.reindex(columns=self.train_columns, fill_value=0)
        return df
    def fit_transform(self, df):
        df = self._encodla_train(df)
        self.train_columns = df.columns 
        return df
    
    def transform(self, df):
        df = self._encodla_test(df)
        return df

prep = Preprocessing()
X_train = prep.fit_transform(X_train)
X_test = prep.transform(X_test)

In [298]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5634, 31)
X_test shape: (1409, 31)


# Predictive (Model-Based) Imputation

In [396]:
for col in X_train.columns:
    # Skip if NaN value is not available
    if (
    X_train[col].isnull().sum() == 0
    and
    X_test[col].isnull().sum() == 0
):
        continue
    # Known and Missing
    known = X_train[X_train[col].notnull()]
    missing = X_train[X_train[col].isnull()]
    # Data for model
    X_known = known.drop(col, axis=1)
    y_known = known[col]

    X_missing = missing.drop(col, axis=1)

    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth = 3,
        learning_rate = 0.01,
        subsample = 1,
        colsample_bytree = 0.8,
        random_state=9
        )
    model.fit(X_known, y_known)
    predicted_values = model.predict(X_missing)
    X_train.loc[X_train[col].isnull(), col] = predicted_values

    if X_test[col].isnull().sum() > 0:

        X_test_missing = X_test[X_test[col].isnull()]

        X_test_missing_features = X_test_missing.drop(col, axis=1)

        predicted_test = model.predict(X_test_missing_features)

        X_test.loc[
            X_test[col].isnull(),
            col
        ] = predicted_test
    


In [300]:
print(X_test.isnull().sum())

SeniorCitizen                            0
tenure                                   0
MonthlyCharges                           0
TotalCharges                             0
yearly_charges                           0
gender_Male                              0
Partner_Yes                              0
Dependents_Yes                           0
PhoneService_Yes                         0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV_No internet service          0
StreamingTV_Yes                          0
StreamingMo

In [301]:
X_train.isnull().sum()

SeniorCitizen                            0
tenure                                   0
MonthlyCharges                           0
TotalCharges                             0
yearly_charges                           0
gender_Male                              0
Partner_Yes                              0
Dependents_Yes                           0
PhoneService_Yes                         0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV_No internet service          0
StreamingTV_Yes                          0
StreamingMo

# Skewness

In [397]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
skewness = X_train[num_cols].skew()
binary_cols = [col for col in num_cols if X_train[col].nunique() == 2]
print(f"Binary columns: {binary_cols}")
encoded_cols = list(prep.label_encoders.keys())
skip_cols = binary_cols + encoded_cols

# skewness > 1.0
high_skew = skewness[(skewness > 1.0) & (~skewness.index.isin(skip_cols))].index
print(f"\nThe columns should be applied for log1p: ")
print(list(high_skew))

for df in [X_train, X_test]:
    for col in high_skew:
        df[col] = np.log1p(df[col])


Binary columns: ['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

The columns should be applied for log1p: 
['SeniorCitizen']


Mid skewness

In [ ]:
# mid_skew = skewness[(skewness >=0.5) & (skewness <=1.0) & (~skewness.index.isin(skip_cols))].index
# print(f"\nColumns should be applied for sqrt: {list(mid_skew)}")

# for df in [X_train, X_test]:
#     for col in mid_skew:
#         df[col] = np.sqrt(df[col])


Columns should be applied for sqrt: []


Negative skewness

In [398]:
neg_skew = skewness[(skewness < -0.5) & (~skewness.index.isin(skip_cols))].index
print(f"\nNegative skewness should be applied for square method: ")
print(list(neg_skew))

for df in [X_train, X_test]:
    for col in neg_skew:
        df[col] = df[col] ** 2
print("\n===Skewness after the tranformation===")
print(X_train[num_cols].skew())


Negative skewness should be applied for square method: 
[]

===Skewness after the tranformation===
SeniorCitizen                            1.845962
tenure                                   0.240016
MonthlyCharges                          -0.226248
TotalCharges                             0.010561
yearly_charges                          -0.226248
gender_Male                             -0.049727
Partner_Yes                              0.061814
Dependents_Yes                           0.880684
PhoneService_Yes                        -2.701997
MultipleLines_No phone service           2.705372
MultipleLines_Yes                        0.302375
InternetService_Fiber optic              0.241057
InternetService_No                       1.384391
OnlineSecurity_No internet service       1.384391
OnlineSecurity_Yes                       0.948661
OnlineBackup_No internet service         1.384391
OnlineBackup_Yes                         0.629875
DeviceProtection_No internet service     1.384391


# High Correlation 

In [399]:
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.8)]
print(f"High Correlation columns to drop: {to_drop}")

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

High Correlation columns to drop: ['yearly_charges', 'MultipleLines_No phone service', 'OnlineSecurity_No internet service', 'OnlineBackup_No internet service', 'DeviceProtection_No internet service', 'TechSupport_No internet service', 'StreamingTV_No internet service', 'StreamingMovies_No internet service']


In [400]:
high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] is not np.nan and upper.loc[row, col] > 0.8:
            high_corr_pairs.append([row, col, round(upper.loc[row, col], 1)])
corr_table = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
print('Highly correlated feature pairs (correlation > 0.8): ')
print(corr_table.to_string(index=False))

Highly correlated feature pairs (correlation > 0.8): 
                           Feature 1                            Feature 2  Correlation
                      MonthlyCharges                       yearly_charges          1.0
                    PhoneService_Yes       MultipleLines_No phone service          1.0
                  InternetService_No   OnlineSecurity_No internet service          1.0
                  InternetService_No     OnlineBackup_No internet service          1.0
  OnlineSecurity_No internet service     OnlineBackup_No internet service          1.0
                  InternetService_No DeviceProtection_No internet service          1.0
  OnlineSecurity_No internet service DeviceProtection_No internet service          1.0
    OnlineBackup_No internet service DeviceProtection_No internet service          1.0
                  InternetService_No      TechSupport_No internet service          1.0
  OnlineSecurity_No internet service      TechSupport_No internet service   

# Heatmap to display among the correlation variables 

In [308]:
corr_long = corr_matrix.reset_index().melt(id_vars = 'index')
corr_long.columns = ['Feature 1', 'Feature 2', 'Correlation']

fig = px.imshow(
    corr_matrix,
    text_auto = '.2f',
    aspect = 'auto',
    color_continuous_scale = 'RdBu_r',
    zmin = 1, zmax = 1,
    title = 'Correlation Matrix Heatmap (Interactive)'
)

fig.update_layout(
    width = 900,
    height = 800,
    xaxis_title = 'Features',
    yaxis_title = 'Features'
)
fig.show()

In [401]:
from sklearn.feature_selection import VarianceThreshold

threshold = 0.01
selector = VarianceThreshold(threshold = threshold)

selector.fit(X_train)
low_features_cols = X_train.columns[~selector.get_support()]
print("===Low Features Drop====")
print(list(low_features_cols))

X_train = X_train.drop(columns=low_features_cols)
X_test = X_test.drop(columns=low_features_cols)

===Low Features Drop====
[]


In [402]:
numerical_cols = X_train.select_dtypes(include=[np.number]).columns
variances = X_train[numerical_cols].var()
threshold = 0.01
low_variance_features = variances[variances<threshold].index.tolist()
var_df = pd.DataFrame({
    'Feature': variances.index,
    'Variances': variances.values,
    'Low Variance': ['Yes' if f in low_variance_features else 'No' for f in variances.index]
})
fig = px.bar(
    data_frame = var_df,
    x = 'Feature',
    y = 'Variances',
    color = 'Low Variance',
    color_discrete_map = {'Yes': 'red', 'No': 'blue'},
    text = 'Variances',
    title = 'Feature Variance ...'
)

fig.update_layout(
    xaxis_tickangle = -45,
    width = 1000,
    height = 600
)
fig.show()

# Embedded Method (Feature Importance)

In [ ]:
# model = xgb.XGBClassifier(
#     random_state=9
# )
# model.fit(X_train, y_train)
# importance_df = pd.DataFrame({
#     'Feature': X_train.columns,
#     'Importance': model.feature_importances_
# })
# importance_df = importance_df.sort_values(
#     by = 'Importance',
#     ascending=False
# )
# print(importance_df)

# fig = px.bar(
#     importance_df,
#     x='Feature',
#     y='Importance',
#     title='XGBoost Feature Importance'
# )

# fig.update_layout(
#     xaxis_tickangle=-45,
#     width=1000,
#     height=600
# )

# fig.show()

                                  Feature  Importance
10            InternetService_Fiber optic    0.284658
19                      Contract_Two year    0.210017
11                     InternetService_No    0.148762
18                      Contract_One year    0.075278
17                    StreamingMovies_Yes    0.032430
2                                  tenure    0.024982
8                        PhoneService_Yes    0.023147
22         PaymentMethod_Electronic check    0.016523
21  PaymentMethod_Credit card (automatic)    0.014749
20                   PaperlessBilling_Yes    0.013743
23             PaymentMethod_Mailed check    0.013454
13                       OnlineBackup_Yes    0.013297
9                       MultipleLines_Yes    0.013118
12                     OnlineSecurity_Yes    0.012490
1                           SeniorCitizen    0.011954
4                            TotalCharges    0.011775
7                          Dependents_Yes    0.011273
3                          M

# Extracting Top Features 

In [ ]:
# top_35_features = int(len(importance_df) * 0.25)
# top_features_35 = importance_df.head(top_35_features)
# selected_features_35 = top_features_35['Feature'].tolist()
# print(f"\n===Selected Top 35% Features===")
# print(selected_features_35)


===Selected Top 35% Features===
['InternetService_Fiber optic', 'Contract_Two year', 'InternetService_No', 'Contract_One year', 'StreamingMovies_Yes', 'tenure']


In [ ]:
# X_train_selected = X_train[selected_features_35]
# X_test_selected = X_test[selected_features_35]

# Scaling --> before applying to l1

In [310]:
X_train.shape


(5634, 23)

In [311]:
X_test.shape

(1409, 23)

In [312]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [315]:
# SMOTE 
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)
print(X_train_smote.shape)
print(y_train_smote.value_counts())

(8276, 23)
Churn
0    4138
1    4138
Name: count, dtype: int64


# L1 Feature Importance 

In [384]:
model = LogisticRegression(C=0.5, max_iter=3000, random_state=42)

model.fit(X_train_smote, y_train_smote)

importance = np.abs(model.coef_[0])

selected_idx = importance > np.percentile(importance, 40)
importance_df = pd.DataFrame({
    "Feature":X_train.columns[selected_idx],
    "Importance":importance[selected_idx]
}).sort_values(by="Importance", ascending=False)
print(f"Top 25% features: {importance_df.to_string(index=False)}")

X_train_selected = X_train_smote[:, selected_idx]
X_test_selected = X_test_scaled[:, selected_idx]

Top 25% features:                        Feature  Importance
                        tenure    0.815327
   InternetService_Fiber optic    0.715476
             Contract_Two year    0.707613
                MonthlyCharges    0.619210
            InternetService_No    0.491354
           StreamingMovies_Yes    0.299890
             Contract_One year    0.298625
               StreamingTV_Yes    0.235353
          PaperlessBilling_Yes    0.203930
             MultipleLines_Yes    0.177992
            OnlineSecurity_Yes    0.162125
PaymentMethod_Electronic check    0.147491
                Dependents_Yes    0.119035
                  TotalCharges    0.116858


In [385]:
final_model = LogisticRegression(
    C=1.0,
    # class_weight='balanced',
    max_iter=3000,
    random_state=42
)

final_model.fit(X_train_selected, y_train_smote)
y_pred_lr = final_model.predict(X_test_selected)

print(f"Classification Report: \n{classification_report(y_test, y_pred_lr)}")
print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred_lr)}")

Classification Report: 
              precision    recall  f1-score   support

           0       0.91      0.78      0.84      1036
           1       0.56      0.78      0.65       373

    accuracy                           0.78      1409
   macro avg       0.73      0.78      0.74      1409
weighted avg       0.82      0.78      0.79      1409

Confusion Matrix: 
[[804 232]
 [ 81 292]]


In [383]:

y_proba = final_model.predict_proba(X_test_selected)[:,1]
threshold = 0.4
y_pred = (y_proba > threshold).astype(int)

print(f"\n=== Logistic Regression | Threshold = {threshold} ===")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


=== Logistic Regression | Threshold = 0.4 ===

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.64      0.76      1036
           1       0.46      0.87      0.60       373

    accuracy                           0.70      1409
   macro avg       0.70      0.75      0.68      1409
weighted avg       0.81      0.70      0.71      1409


Confusion Matrix:

[[658 378]
 [ 49 324]]


# Alogorithms need to be scaled 

In [387]:
# Using Pipelines and joblib to increase performance
from joblib import Parallel, delayed
pipelines = {
    "LR": LogisticRegression(random_state=9, class_weight='balanced', max_iter=1000),
    "SVM": SVC(kernel='linear', C=1.0),
    "KNN": KNeighborsClassifier()
}

def train_model(name, model):
    model.fit(X_train_selected, y_train_smote)
    y_pred_scaled = model.predict(X_test_selected)
    return name, model, y_pred_scaled

results_scaled_improvement = Parallel(n_jobs=-1)(
    delayed(train_model)(name, model)
    for name, model in pipelines.items()
)
# Results 
for name, model, y_pred_scaled in results_scaled_improvement:
    print(f"\n==={name}===")
    print(f"Classification Report: \n{classification_report(y_test, y_pred_scaled)}")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred_scaled)}")


===LR===
Classification Report: 
              precision    recall  f1-score   support

           0       0.91      0.78      0.84      1036
           1       0.56      0.78      0.65       373

    accuracy                           0.78      1409
   macro avg       0.73      0.78      0.74      1409
weighted avg       0.82      0.78      0.79      1409

Confusion Matrix: 
[[804 232]
 [ 81 292]]

===SVM===
Classification Report: 
              precision    recall  f1-score   support

           0       0.93      0.65      0.77      1036
           1       0.47      0.86      0.61       373

    accuracy                           0.71      1409
   macro avg       0.70      0.76      0.69      1409
weighted avg       0.81      0.71      0.72      1409

Confusion Matrix: 
[[674 362]
 [ 52 321]]

===KNN===
Classification Report: 
              precision    recall  f1-score   support

           0       0.87      0.74      0.80      1036
           1       0.49      0.68      0.57      

# Model Training 

# Alogrithms no need to be scaled


In [403]:
# SMOTE 
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)
print(X_train_smote.shape)
print(y_train_smote.value_counts())

(8276, 23)
Churn
0    4138
1    4138
Name: count, dtype: int64


In [409]:
from sklearn.pipeline import Pipeline
from joblib import Parallel, delayed
pipelines = {
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200,
        max_depth = 3,
        learning_rate = 0.01,
        subsample = 1,
        colsample_bytree = 0.8,
        random_state=9
    )
}

def train_model(name, model):
    model.fit(X_train_smote, y_train_smote)
    y_pred_improvement = model.predict(X_test)
    return name, model, y_pred_improvement

results_no_scaled = Parallel(n_jobs=-1)(
    delayed(train_model)(name, model)
    for name, model in pipelines.items()
)
# Results 
for name, model, y_pred_improvement in results_no_scaled:
    print(f"\n==={name}===")
    print(f"Classification Report: \n{classification_report(y_test, y_pred_improvement)}")
    print(f"Confusion Matrix: \n{confusion_matrix(y_test, y_pred_improvement)}")
    print(recall_score(y_test, y_pred_improvement))


===Decision Tree===
Classification Report: 
              precision    recall  f1-score   support

           0       0.80      0.76      0.78      1036
           1       0.42      0.47      0.44       373

    accuracy                           0.69      1409
   macro avg       0.61      0.62      0.61      1409
weighted avg       0.70      0.69      0.69      1409

Confusion Matrix: 
[[792 244]
 [198 175]]
0.4691689008042895

===Random Forest===
Classification Report: 
              precision    recall  f1-score   support

           0       0.85      0.86      0.85      1036
           1       0.59      0.57      0.58       373

    accuracy                           0.78      1409
   macro avg       0.72      0.71      0.71      1409
weighted avg       0.78      0.78      0.78      1409

Confusion Matrix: 
[[889 147]
 [162 211]]
0.5656836461126006

===XGBoost===
Classification Report: 
              precision    recall  f1-score   support

           0       0.92      0.74      0

# Comparison model results among six algorithms 

In [413]:
from tabulate import tabulate

improvement = [
    ['Decision Tree Classifier',0.69, 0.47],
    ['Random Forest Classifier', 0.78, 0.57],
    ['XGBClassifier',0.76, 0.81],
    ['Logistic Regression',0.78, 0.78],
    ['Support Vector Machine',0.71, 0.86],
    ['KNN', 0.73, 0.68]
    
]
headers = ['Models', 'Accuracy', 'Recall']
table = tabulate(improvement, headers=headers, tablefmt='grid', floatfmt='.4f')
print('\n===============================================================================')
print(table)
print('===============================================================================')


+--------------------------+------------+----------+
| Models                   |   Accuracy |   Recall |
+==========================+============+==========+
| Decision Tree Classifier |     0.6900 |   0.4700 |
+--------------------------+------------+----------+
| Random Forest Classifier |     0.7800 |   0.5700 |
+--------------------------+------------+----------+
| XGBClassifier            |     0.7600 |   0.8100 |
+--------------------------+------------+----------+
| Logistic Regression      |     0.7800 |   0.7800 |
+--------------------------+------------+----------+
| Support Vector Machine   |     0.7100 |   0.8600 |
+--------------------------+------------+----------+
| KNN                      |     0.7300 |   0.6800 |
+--------------------------+------------+----------+


In [414]:
from tabulate import tabulate

baseline = [
    ['Decision Tree Classifier',0.68, 0.32 ],
    ['Random Forest Classifier', 0.78, 0.47],
    ['XGBClassifier',0.79, 0.36],
    ['Logistic Regression',0.80, 0.53],
    ['Support Vector Machine',0.80, 0.53],
    ['KNN', 0.75, 0.48]
    
]
headers = ['Models', 'Accuracy', 'Recall']
table = tabulate(baseline, headers=headers, tablefmt='grid', floatfmt='.4f')
print('\n===============================================================================')
print(table)
print('===============================================================================')


+--------------------------+------------+----------+
| Models                   |   Accuracy |   Recall |
+==========================+============+==========+
| Decision Tree Classifier |     0.6800 |   0.3200 |
+--------------------------+------------+----------+
| Random Forest Classifier |     0.7800 |   0.4700 |
+--------------------------+------------+----------+
| XGBClassifier            |     0.7900 |   0.3600 |
+--------------------------+------------+----------+
| Logistic Regression      |     0.8000 |   0.5300 |
+--------------------------+------------+----------+
| Support Vector Machine   |     0.8000 |   0.5300 |
+--------------------------+------------+----------+
| KNN                      |     0.7500 |   0.4800 |
+--------------------------+------------+----------+


In [417]:
from tabulate import tabulate

comparison_results = []

for model_name,_, model_recall in improvement:
    baseline_recall = baseline_dict.get(model_name)

    diff = model_recall - baseline_recall

    if diff > 0:
        status = f"+{diff:.2f} 🟢 improved"
    elif diff < 0:
        status = f"{diff:.2f} 🔴 decreased"
    else:
        status = "0.00 ⚪ stable"

    comparison_results.append([
        model_name,
        baseline_recall,
        model_recall,
        diff,
        status
    ])

headers = ['Model', 'Baseline Recall', 'Improved Recall', 'Diff', 'Status']

print(tabulate(comparison_results, headers=headers, tablefmt='grid', floatfmt='.2f'))

+--------------------------+-------------------+-------------------+--------+-------------------+
| Model                    |   Baseline Recall |   Improved Recall |   Diff | Status            |
+==========================+===================+===================+========+===================+
| Decision Tree Classifier |              0.33 |              0.47 |   0.14 | +0.14 🟢 improved |
+--------------------------+-------------------+-------------------+--------+-------------------+
| Random Forest Classifier |              0.47 |              0.57 |   0.10 | +0.10 🟢 improved |
+--------------------------+-------------------+-------------------+--------+-------------------+
| XGBClassifier            |              0.36 |              0.81 |   0.45 | +0.45 🟢 improved |
+--------------------------+-------------------+-------------------+--------+-------------------+
| Logistic Regression      |              0.53 |              0.78 |   0.25 | +0.25 🟢 improved |
+-----------------------

In [ ]:
"""

Bu yerda men SMOTE dan fodyalandim sababi datasetim inbalance edi.
Men uchun asosiy vazifa bu yerda recallni balandor qilish edi. 
Sababi "Churn" Bu yerda; No  = 0  → customer qoladi
Yes = 1  → customer ketadi (leaves company)
Agar customer ketsa, revenue kamayadi. Marketing cost oshadi.
Yes --> y'ani qancha recall baland bo'lsa shuncha kompaniya ularga bog'lanib,
special offer, discount va call center orqali ushlab qoladi.

"""


"""
Ko'rib turganingizdek Eng katta o'sish recallda XGBClassifierda 
baselineda: 0.36 edi ammo improvementda so'ng: 0.81 + 0.45% yaxshilanish.(bu degani har 100tadan 81 tani topayapti)
Undan keyin SVMda unda 0.53 dan ti 0.86 gacha o'sgan va eng yuqori recallni tashkil edi.
Bu degani har 100ta dan 86 ta odamni aniq topayapti va ularga maxsus offer va discountlar taqdim qiladi.

"""

# Best Model 

In [418]:
from joblib import dump, load
dump('y_pred_scaled', 'support_vector_machine_classifier.joblib')

['support_vector_machine_classifier.joblib']